# AIC2026 — extract keyframes + CLIP gallery

1. Edit **`VIDEO_DIRS`**: folders of `.mp4`
2. Extract → flat `KEYFRAMES_OUT/VIDEO_ID/{map.csv,*.webp,embeddings.npy}`
3. Organize index artifacts:
   - `features/clip/L**/VIDEO_ID.npy`
   - `features/maps/VIDEO_ID.csv`
4. Zip for Output download (ready to upload as Kaggle datasets):
   - `keyframes-L**.zip` — archive root = `VIDEO_ID/...` (same layout as uploaded L21)
   - **`features.zip`** — whole `clip/` + `maps/` (easiest index download)
   - `features-clip-L**.zip` / `features-maps.zip` — optional splits

Clone + pip first; enable GPU. Save Version → download `*.zip` from Output.


In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("AIC2026-PACs-token")


In [ ]:
!rm -rf /kaggle/working/AIC2026-PACs
!git clone https://{secret_value_0}@github.com/AkiyaNguyen/AIC2026-PACs.git


In [ ]:
!pip install -q -r /kaggle/working/AIC2026-PACs/requirements.txt


In [ ]:
# Each folder of mp4s. Output uses **video file stem** as folder name (NOT the
# parent shard folder). So L26 split across a..e still merges into one batch:
#   .../Videos_L26_a/L26_V001.mp4 → KEYFRAMES_OUT/L26_V001/...
#   .../Videos_L26_e/L26_V099.mp4 → KEYFRAMES_OUT/L26_V099/...
# Later: features/clip/L26/, keyframes-L26.zip  (grouped by L\d+ from the stem)
VIDEO_DIRS = [
    # "/kaggle/input/datasets/akiyanguyen/pacs-data/video",
    # "/kaggle/input/datasets/akiyanguyen/pacs-data-l22-a/video",
    # "/kaggle/input/datasets/akiyanguyen/pacs-data-l23-a/video",
    # "/kaggle/input/datasets/akiyanguyen/pacs-data-l24-a/video",
    # "/kaggle/input/datasets/akiyanguyen/pacs-data-l25-a/video",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_a",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_b",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_c",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_d",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l26/Videos_L26_e",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l27-a/video",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l28-a/video",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l29/video",
    "/kaggle/input/datasets/akiyanguyen/pacs-data-l30-a/video",
]


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO = "/kaggle/working/AIC2026-PACs"
KEYFRAMES_OUT = "/kaggle/working/keyframes-out"
# Flat embed staging (then split into features/clip/L**/)
EMBED_OUT = "/kaggle/working/clip-gallery"
FEATURES_ROOT = "/kaggle/working/features"
FEATURES_CLIP = f"{FEATURES_ROOT}/clip"
FEATURES_MAPS = f"{FEATURES_ROOT}/maps"

STRIDE = 10
MIN_COSINE_DISTANCE = 0.15
BATCH_SIZE = 32
DEVICE = "gpu"  # cpu | gpu | cuda
# Batches to zip as keyframes-L**.zip / features-clip-L**.zip (L21..L30)
BATCH_IDS = [f"L{n}" for n in range(21, 31)]

VIDEO_DIRS = [str(Path(d)) for d in VIDEO_DIRS]
missing = [d for d in VIDEO_DIRS if not Path(d).is_dir()]
if missing:
    raise SystemExit(f"Not a directory: {missing}")

for d in VIDEO_DIRS:
    n = len(list(Path(d).glob("*.mp4")))
    if n == 0:
        n = len(list(Path(d).rglob("*.mp4")))
    print(f"{d}  mp4_count={n}")

for k, v in {
    "REPO": REPO,
    "KEYFRAMES_OUT": KEYFRAMES_OUT,
    "EMBED_OUT": EMBED_OUT,
    "FEATURES_ROOT": FEATURES_ROOT,
    "FEATURES_CLIP": FEATURES_CLIP,
    "FEATURES_MAPS": FEATURES_MAPS,
    "STRIDE": str(STRIDE),
    "MIN_COSINE_DISTANCE": str(MIN_COSINE_DISTANCE),
    "BATCH_SIZE": str(BATCH_SIZE),
    "DEVICE": DEVICE,
}.items():
    os.environ[k] = v

print("KEYFRAMES_OUT =", KEYFRAMES_OUT)
print("FEATURES_CLIP =", FEATURES_CLIP)
print("FEATURES_MAPS =", FEATURES_MAPS)


## Extract

All dirs write into the **same** flat tree (video stem = folder name). Shard folders like `Videos_L26_a` … `_e` do **not** become separate outputs — only the mp4 stem matters (`L26_V001` → batch **L26**).

```text
# inputs (example)
.../Videos_L26_a/L26_V001.mp4
.../Videos_L26_e/L26_V050.mp4

# after extract (merged)
keyframes-out/
  L26_V001/{map.csv, *.webp, embeddings.npy}
  L26_V050/...
  L27_V001/...
```

Same rule for zips: one `keyframes-L26.zip` and `features/clip/L26/`.


In [ ]:
import re
from collections import Counter

for i, video_dir in enumerate(VIDEO_DIRS, start=1):
    print(f"========== [{i}/{len(VIDEO_DIRS)}] {video_dir} ==========", flush=True)
    cmd = [
        sys.executable, "-m", "tools.extract_features", "extract", video_dir,
        "--out-dir", KEYFRAMES_OUT,
        "--stride", str(STRIDE),
        "--min-cosine-distance", str(MIN_COSINE_DISTANCE),
        "--batch-size", str(BATCH_SIZE),
        "--device", DEVICE,
    ]
    subprocess.run(cmd, cwd=REPO, check=True)

vids = sorted(p.name for p in Path(KEYFRAMES_OUT).iterdir() if p.is_dir())
print("Done extract →", KEYFRAMES_OUT, f"({len(vids)} VIDEO_ID folders)")

# Confirm shards (e.g. L26_a..e) collapsed by stem prefix L\d+
batch_re = re.compile(r"^(L\d+)_V\d+$")
counts = Counter()
unknown = []
for v in vids:
    m = batch_re.match(v)
    if m:
        counts[m.group(1)] += 1
    else:
        unknown.append(v)
print("Videos per batch (from stem, not input folder):")
for b, n in sorted(counts.items()):
    print(f"  {b}: {n}")
if unknown:
    print("WARNING — unexpected VIDEO_ID names (won't join L** batch):", unknown[:20])
print(vids[:15], "...")


## Embed + organize index layout

1. `embed --copy-embeddings` → flat staging `EMBED_OUT/VIDEO_ID.npy`
2. Split into repo layout:
   - `features/clip/L21/L21_V001.npy` …
   - `features/maps/L21_V001.csv` (from each `VIDEO_ID/map.csv`)


In [ ]:
import json
import re
import shutil

cmd = [
    sys.executable, "-m", "tools.extract_features", "embed", KEYFRAMES_OUT,
    "--out-dir", EMBED_OUT,
    "--copy-embeddings",
]
subprocess.run(cmd, cwd=REPO, check=True)

kf = Path(KEYFRAMES_OUT)
gal = Path(EMBED_OUT)
clip_root = Path(FEATURES_CLIP)
maps_root = Path(FEATURES_MAPS)
clip_root.mkdir(parents=True, exist_ok=True)
maps_root.mkdir(parents=True, exist_ok=True)

batch_re = re.compile(r"^(L\d+)_V\d+$")
by_batch: dict[str, list[str]] = {}

for npy in sorted(gal.glob("*.npy")):
    video_id = npy.stem
    m = batch_re.match(video_id)
    if not m:
        print(f"skip unexpected npy name: {npy.name}")
        continue
    batch = m.group(1)
    dest_dir = clip_root / batch
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(npy, dest_dir / npy.name)
    by_batch.setdefault(batch, []).append(video_id)

    map_src = kf / video_id / "map.csv"
    if map_src.is_file():
        shutil.copy2(map_src, maps_root / f"{video_id}.csv")
    else:
        print(f"warning: missing map.csv for {video_id}")

# Preserve / refresh run_meta per batch (from flat embed meta if present)
flat_meta_path = gal / "run_meta.json"
flat_meta = {}
if flat_meta_path.is_file():
    flat_meta = json.loads(flat_meta_path.read_text(encoding="utf-8"))

for batch, vids in sorted(by_batch.items()):
    videos = []
    for video_id in sorted(vids):
        entry = {"video_id": video_id, "npy": f"{video_id}.npy", "source": "embeddings.npy"}
        if flat_meta.get("videos"):
            hit = next((v for v in flat_meta["videos"] if v.get("video_id") == video_id), None)
            if hit:
                entry.update({k: hit[k] for k in ("n_rows", "dim") if k in hit})
        videos.append(entry)
    meta = {
        "model": flat_meta.get("model", "copy:embeddings.npy"),
        "device": flat_meta.get("device"),
        "batch_size": flat_meta.get("batch_size"),
        "copied": True,
        "n_videos": len(videos),
        "videos": videos,
    }
    (clip_root / batch / "run_meta.json").write_text(
        json.dumps(meta, indent=2), encoding="utf-8"
    )

print("features/clip batches:", sorted(p.name for p in clip_root.iterdir() if p.is_dir()))
print("features/maps csv:", len(list(maps_root.glob("*.csv"))))


## Zip for Kaggle Output (upload-ready)

| Zip | Archive layout | Use |
|-----|----------------|-----|
| `keyframes-L21.zip` | `L21_V001/{map.csv,*.webp,embeddings.npy}/…` at **zip root** | Manual dataset upload (same as existing L21) |
| **`features.zip`** | `clip/L**/VIDEO_ID.npy` + `maps/VIDEO_ID.csv` | **One download** → unpack into repo `features/` |
| `features-clip-L21.zip` | `L21/L21_V001.npy` + `run_meta.json` | Per-batch gallery (optional) |
| `features-maps.zip` | `L21_V001.csv` … (flat) | `kis_search --map-dir` |

Also writes `dataset-metadata-L**.json` beside each keyframes zip (for `kaggle datasets create`).

After Save Version: `kaggle kernels output USER/slug -p dest --file-pattern '\.zip$'`


In [ ]:
import json
import zipfile
from IPython.display import display, FileLink, Markdown

work = Path("/kaggle/working")
kf = Path(KEYFRAMES_OUT)
features_root = Path(FEATURES_ROOT)
clip_root = Path(FEATURES_CLIP)
maps_root = Path(FEATURES_MAPS)

KAGGLE_USER = "akiyanguyen"  # dataset id prefix


def write_dataset_metadata(batch: str, out_json: Path) -> dict:
    meta = {
        "title": f"AIC2026 Keyframes {batch}",
        "id": f"{KAGGLE_USER}/aic2026-keyframes-{batch.lower()}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    out_json.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return meta


# 1) keyframes-L**.zip — VIDEO_ID/ at archive root (matches uploaded L21 layout)
ZIP_STILLS = True
if ZIP_STILLS:
    for batch in BATCH_IDS:
        prefix = f"{batch}_"
        dirs = sorted(p for p in kf.iterdir() if p.is_dir() and p.name.startswith(prefix))
        if not dirs:
            continue
        out = work / f"keyframes-{batch}.zip"
        out.unlink(missing_ok=True)
        meta = write_dataset_metadata(batch, work / f"dataset-metadata-{batch}.json")
        nfiles = 0
        with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as zf:
            zf.writestr("dataset-metadata.json", json.dumps(meta, indent=2))
            for d in dirs:
                for f in d.rglob("*"):
                    if f.is_file():
                        # L21_V001/map.csv  (NOT keyframes-out/L21_V001/...)
                        zf.write(f, arcname=str(f.relative_to(kf)))
                        nfiles += 1
        print(f"Wrote {out.name}: {out.stat().st_size/1e6:.1f} MB  ({nfiles} files, {len(dirs)} videos)")

# 2) features.zip — whole tree (clip/ + maps/) for one-shot download
feat_files = [p for p in features_root.rglob("*") if p.is_file()]
if feat_files:
    out = work / "features.zip"
    out.unlink(missing_ok=True)
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for f in feat_files:
            # clip/L21/L21_V001.npy , maps/L21_V001.csv
            zf.write(f, arcname=str(f.relative_to(features_root)))
    print(f"Wrote {out.name}: {out.stat().st_size/1e6:.1f} MB  ({len(feat_files)} files)")

# 3) features-clip-L**.zip — optional per-batch
for batch_dir in sorted(p for p in clip_root.iterdir() if p.is_dir()):
    files = [p for p in batch_dir.rglob("*") if p.is_file()]
    if not files:
        continue
    out = work / f"features-clip-{batch_dir.name}.zip"
    out.unlink(missing_ok=True)
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for f in files:
            zf.write(f, arcname=str(f.relative_to(clip_root)))
    print(f"Wrote {out.name}: {out.stat().st_size/1e6:.1f} MB  ({len(files)} files)")

# 4) features-maps.zip — flat VIDEO_ID.csv
map_files = sorted(maps_root.glob("*.csv"))
if map_files:
    out = work / "features-maps.zip"
    out.unlink(missing_ok=True)
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for f in map_files:
            zf.write(f, arcname=f.name)
    print(f"Wrote {out.name}: {out.stat().st_size/1e6:.1f} MB  ({len(map_files)} csv)")

display(Markdown("### Download zips (click, or Save Version then `kaggle kernels output`)"))
for z in sorted(work.glob("*.zip")):
    print(f"{z.name}: {z.stat().st_size/1e6:.1f} MB")
    display(FileLink(str(z)))
